# Phase 3: Experiment Tracking
## DNA Gene Mapping Project
**Author:** Sharique Mohammad
**Date:** February 2026

---

## Objective
Track all ML experiments in PostgreSQL:
- Create ml_experiments table
- Log all model training runs
- Track hyperparameters and metrics
- Enable model comparison

## Why Experiment Tracking
- Compare models over time
- Reproduce results
- ML ops best practice
- Audit trail

In [ ]:
import pandas as pd
import json
from datetime import datetime
import psycopg2
from pathlib import Path

print("Experiment Tracking Setup")
print("="*80)

## Create Experiments Table

In [ ]:
# Connect to PostgreSQL using config
try:
    # Import config
    import sys
    from pathlib import Path
    
    # Add project root to path
    project_root = Path.cwd().parent.parent
    sys.path.insert(0, str(project_root))
    
    from config import DATABASE_CONFIG
    
    # Connect using config
    conn = psycopg2.connect(
        dbname=DATABASE_CONFIG['database'],
        user=DATABASE_CONFIG['user'],
        password=DATABASE_CONFIG['password'],
        host=DATABASE_CONFIG['host'],
        port=DATABASE_CONFIG['port']
    )
    cursor = conn.cursor()
    
    # Create experiments table
    create_table = """
    CREATE TABLE IF NOT EXISTS ml_experiments (
        experiment_id SERIAL PRIMARY KEY,
        experiment_name VARCHAR(255),
        model_type VARCHAR(100),
        task VARCHAR(100),
        algorithm VARCHAR(100),
        hyperparameters JSONB,
        training_samples INTEGER,
        test_samples INTEGER,
        n_features INTEGER,
        test_f1 FLOAT,
        test_precision FLOAT,
        test_recall FLOAT,
        test_roc_auc FLOAT,
        training_time_seconds INTEGER,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        notes TEXT
    );
    """
    
    cursor.execute(create_table)
    conn.commit()
    print("ml_experiments table created using credentials from config.py")
    
except Exception as e:
    print(f"Note: Could not connect to PostgreSQL: {e}")
    print("Skipping database operations. Logging to JSON instead.")
    conn = None

## Log Variant Model Experiments

In [ ]:
experiments = [
    {
        "experiment_name": "variant_baseline_lr",
        "model_type": "variant_pathogenicity",
        "task": "binary_classification",
        "algorithm": "LogisticRegression",
        "hyperparameters": {"max_iter": 1000},
        "training_samples": 1205635,
        "test_samples": 623435,
        "n_features": 75,
        "test_f1": 0.7617,
        "test_precision": 0.6688,
        "test_recall": 0.8845,
        "test_roc_auc": 0.9803,
        "training_time_seconds": 60,
        "notes": "Baseline model"
    },
    {
        "experiment_name": "variant_xgboost_tuned",
        "model_type": "variant_pathogenicity",
        "task": "binary_classification",
        "algorithm": "XGBoost",
        "hyperparameters": {
            "n_estimators": 300,
            "max_depth": 7,
            "learning_rate": 0.1
        },
        "training_samples": 1205635,
        "test_samples": 623435,
        "n_features": 75,
        "test_f1": 0.8363,
        "test_precision": 0.7946,
        "test_recall": 0.8827,
        "test_roc_auc": 0.9894,
        "training_time_seconds": 1884,
        "notes": "Production model - best performance"
    },
    {
        "experiment_name": "sv_xgboost_raw_features",
        "model_type": "sv_risk",
        "task": "binary_classification",
        "algorithm": "XGBoost",
        "hyperparameters": {
            "n_estimators": 100,
            "max_depth": 7,
            "learning_rate": 0.05
        },
        "training_samples": 151948,
        "test_samples": 32512,
        "n_features": 8,
        "test_f1": 0.9783,
        "test_precision": 0.9772,
        "test_recall": 0.9794,
        "test_roc_auc": 0.9983,
        "training_time_seconds": 300,
        "notes": "Raw features only - no derived scores"
    }
]

if conn:
    for exp in experiments:
        insert_query = """
        INSERT INTO ml_experiments (
            experiment_name, model_type, task, algorithm,
            hyperparameters, training_samples, test_samples,
            n_features, test_f1, test_precision, test_recall,
            test_roc_auc, training_time_seconds, notes
        ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        """
        
        cursor.execute(insert_query, (
            exp['experiment_name'],
            exp['model_type'],
            exp['task'],
            exp['algorithm'],
            json.dumps(exp['hyperparameters']),
            exp['training_samples'],
            exp['test_samples'],
            exp['n_features'],
            exp['test_f1'],
            exp['test_precision'],
            exp['test_recall'],
            exp['test_roc_auc'],
            exp['training_time_seconds'],
            exp['notes']
        ))
    
    conn.commit()
    print(f"Logged {len(experiments)} experiments to PostgreSQL")
else:
    # Fallback: Save to JSON
    PROJECT_ROOT = Path.cwd().parent.parent
    with open(PROJECT_ROOT / "data" / "ml" / "experiments.json", 'w') as f:
        json.dump(experiments, f, indent=2)
    print(f"Logged {len(experiments)} experiments to experiments.json")

## Query Experiments

In [ ]:
if conn:
    # Query best models by task
    query = """
    SELECT 
        model_type,
        experiment_name,
        algorithm,
        test_f1,
        test_roc_auc
    FROM ml_experiments
    ORDER BY model_type, test_f1 DESC
    """
    
    df = pd.read_sql(query, conn)
    print("\nAll Experiments:")
    print(df.to_string(index=False))
    
    cursor.close()
    conn.close()
else:
    print("\nExperiments logged to JSON file (PostgreSQL not available)")

In [ ]:
print("\n" + "="*80)
print("EXPERIMENT TRACKING COMPLETE")
print("="*80)
print("\nLogged experiments:")
for exp in experiments:
    print(f"  - {exp['experiment_name']}: F1={exp['test_f1']:.4f}")
print("\nExperiments can now be queried from ml_experiments table")
print("="*80)